# Pattern Detection – Day 7: Data & Domain

**Student Name:** Traver Dinten
**Country:** Switzerland
**Semester term:** FS26

## Use Case and Application Context

In the context of satellite-based glacier monitoring, Sentinel-2 imagery of the Aletsch Glacier is used to identify structural surface patterns that indicate glacier dynamics and retreat behaviour. The Aletsch Glacier — the largest glacier in the Alps — exhibits pronounced linear surface features: medial moraines formed by the convergence of tributary glaciers, lateral moraine boundaries separating glacier ice from surrounding bedrock, and the glacier terminus boundary marking the current extent of the ice body.

These linear structures are key indicators used by glaciologists to assess glacier flow, mass balance changes, and retreat rates. Detecting such boundary and moraine features in satellite imagery supports long-term environmental monitoring and has direct implications for Swiss water resource management, natural hazard assessment, and Alpine infrastructure planning.

The use case targets **edge detection** — specifically the identification of intensity transitions that correspond to physical boundaries on the glacier surface. This is coherently aligned with the Canny edge detection algorithm implemented in Day 6.

## Problem Statement

At the 10 m spatial resolution of Sentinel-2, glacier surface structures such as moraine lines and ice-rock boundaries appear as subtle intensity transitions spanning only a few pixels. The challenge is that the image simultaneously contains:

- **True structural edges** — moraine boundaries, ice-rock transitions, and the glacier terminus — that carry geologically meaningful information
- **Surface texture variations** — crevasse fields, debris cover patterns, and snow grain differences — that produce intensity gradients indistinguishable in magnitude from real structural edges

If edge detection is applied without appropriate parameterization, the result may either miss critical structural boundaries (under-detection) or produce excessive false edges from surface texture (over-detection), rendering the resulting edge map unreliable for glacier monitoring purposes.

## Image Selection and Description

The selected image is a **Sentinel-2 Level-2A Surface Reflectance composite** of the Aletsch Glacier tongue region, acquired during summer 2023 (June–September median composite). The composite uses bands B4 (Red), B3 (Green), and B2 (Blue) at 10 m ground sampling distance.

**Origin:** Copernicus Sentinel-2 via Google Earth Engine (`COPERNICUS/S2_SR_HARMONIZED`), freely available under the Copernicus open data policy. The image is cached locally in `data/raw/aletsch_tongue_s2.tif`.

**Content:** The image covers the lower tongue of the Aletsch Glacier, including the glacier terminus, lateral valley walls, and the proglacial area. The glacier tongue region is specifically chosen because it exhibits the highest density of visible linear features — medial moraines running along the flow direction, lateral moraine boundaries, and the terminus — within a single coherent scene.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join("..", "helpers"))

import numpy as np
import matplotlib.pyplot as plt
from gee_utils import load_image

DATA_DIR = os.path.join("..", "data", "raw")
FIGURES_DIR = os.path.join("..", "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

IMAGE_PATH = os.path.join(DATA_DIR, "aletsch_tongue_s2.tif")
img_rgb = load_image(IMAGE_PATH)
print(f"Image shape: {img_rgb.shape}")
print(f"Pixel value range: [{img_rgb.min()}, {img_rgb.max()}]")
print(f"Data type: {img_rgb.dtype}")

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 7))
ax.imshow(img_rgb)
ax.set_title("Sentinel-2 – Aletsch Glacier Tongue (Summer 2023)", fontsize=13)
ax.set_xlabel("Column (pixel)")
ax.set_ylabel("Row (pixel)")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "day07_original.png"), dpi=150, bbox_inches="tight")
plt.show()

## Visual Characteristics

The image exhibits the following visual characteristics relevant to edge detection:

- **High-contrast boundaries:** The ice-rock transitions along the lateral valley walls produce sharp intensity changes (bright ice/snow vs. dark rock) spanning 1–3 pixels at 10 m resolution. These are the strongest edges in the scene.
- **Medial moraine lines:** Dark linear features running parallel to the glacier flow direction, formed by debris bands from tributary glaciers. These moraines create moderate-contrast edges against the brighter surrounding ice.
- **Surface texture:** The glacier surface contains spatially varying texture from crevasses, meltwater channels, and debris cover. These produce numerous low-to-moderate intensity gradients that may be confused with structural edges.
- **Terminus boundary:** The glacier terminus shows a transition from ice to proglacial sediment/rock, forming a curved boundary with variable contrast depending on local debris coverage.
- **Illumination gradients:** Terrain shadows from the surrounding valley walls create large-scale intensity gradients across the scene that are not related to glacier surface structures.

## Pattern of Interest

The pattern of interest is **linear edge structures** — specifically the intensity transitions that correspond to:

1. **Glacier-rock boundaries** — the lateral edges where glacier ice meets valley walls
2. **Medial moraines** — dark linear debris bands running along the glacier flow
3. **Terminus boundary** — the curved edge marking the current glacier extent

In this Sentinel-2 image, these patterns manifest as **local intensity discontinuities** where adjacent pixels show a measurable difference in surface reflectance. The moraine lines appear as elongated dark streaks (lower reflectance due to rock debris) against the brighter glacier ice (higher reflectance from snow/firn). The ice-rock boundaries appear as abrupt transitions between the spectrally distinct surfaces of glacier ice and exposed bedrock or vegetation.

These edges are well-suited for gradient-based detection because they represent physically meaningful transitions, not artifacts. However, their detectability depends on the edge strength relative to the surrounding surface texture.

## Importance of Detection and Objective

Detecting these edge structures is important for the defined glacier monitoring use case because:

- **Glacier extent mapping:** The lateral and terminus boundaries define the spatial extent of the glacier. Tracking these edges over time enables quantification of retreat rates — the Aletsch Glacier has retreated over 3.5 km since the Little Ice Age maximum.
- **Flow structure analysis:** Medial moraine positions and geometry reflect the flow dynamics of the glacier. Shifts in moraine alignment indicate changes in tributary contributions and flow velocity.
- **Automated monitoring:** Manual digitization of glacier boundaries from satellite imagery is time-consuming. Reliable edge detection provides a basis for semi-automated boundary extraction applicable across large image archives.

**Objective:** The objective of pattern detection in this context is to extract edge maps from Sentinel-2 imagery that reliably capture the glacier's structural boundaries (moraine lines, ice-rock edges, terminus) while suppressing responses from surface texture and illumination gradients. The quality of this extraction depends on the parameterization of the edge detection algorithm — specifically the balance between noise suppression and edge sensitivity — which will be investigated in subsequent days.